<center><a target="_blank" href="https://academy.constructor.org/"><img src=https://lh3.googleusercontent.com/d/1fypIr9T-7ntcsVQFmC2_iMPcsm7h8jXg width="500" style="background:none; border:none; box-shadow:none;" /></a> </center>
<hr />

# <h1 align="center"> Day-2, Exercise 1: Point Estimates and Confidence Intervals </h1> </center>

<p style="margin-bottom:1cm;"></p>

_____

<center>Constructor Nexademy, 2026</center>




Let's have a look at the Marketing dataset of yesterday:

In [2]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as st

In [4]:
marketing_data = pd.read_csv('/home/soraan/Desktop/data/Marketing3.csv')
marketing_data.head(6)

,Unnamed: 0,recency,history,used_discount,used_bogo,zip_code,is_referral,channel,offer,conversion
0,0,10,142.44,1,0,Surburban,0,Phone,Buy One Get One,0
1,1,5,349.41,1,1,Surburban,1,Phone,Buy One Get One,0
2,2,7,64.04,0,1,Rural,0,Web,Buy One Get One,1
3,3,2,244.82,0,1,Rural,1,Web,Buy One Get One,1
4,4,11,302.15,1,0,Urban,0,Web,Buy One Get One,0
5,5,1,525.05,1,0,Surburban,1,Multichannel,Buy One Get One,0


Let's focus on the "history" column. As you may remember, it stores the total
amount ever spent by the customer in the shop.

In this exercise, we are going to play with the concept of sample mean and how it
relates to population mean. The main idea behind the exercise is to train
how to compute estimators and confidence initervals when we only have a small
sample of a very large population. In our example this amounts to the following
problem:

suppose that instead of a whole database with customer history, we only have a
data on the history of 20 customers. Can we estimate the average history of the
whole population from this sample? How sure are we about this value?

We start by classical point estimation and confidence intervals (Part 1),
and then move on to applying the bootstrapping method (Part 2).

# Part 1. Point Estimates And confidence intervals

**FYI The following can also be done with only margin of error - the solutions show that approach.**

1. Sample 20 values from the history column



<!--- BEGIN SOLUTION --->

In [10]:
# your code here

sample_data = marketing_data['history'].sample(20)
print(sample_data)

53620      36.73
6829      345.98
34547     494.38
11962     451.64
11402      98.31
32154    2728.69
62619      29.99
15974     183.58
18163      29.99
20929     383.90
30048     324.28
37436     148.04
35076      55.15
15347      29.99
61220      29.99
59429     518.38
22205     150.79
30283      29.99
20065     360.77
56897     506.56
Name: history, dtype: float64


<!--- END SOLUTION --->

2. Use this sample to compute the sample mean and 95 confidence interval from the formula shown in the lecture slides for calculating the sample mean & CI.

<!--- BEGIN SOLUTION --->

In [11]:
# your code here


from scipy.stats import norm

confidence_level = 0.95

z_critical = norm.ppf(1 - (1 - confidence_level) / 2)

sample_mean = sample_data.mean()
sample_std = sample_data.std()
n = 20

standard_error = sample_std / np.sqrt(n)
margin_of_error = z_critical * standard_error

ci_lower = sample_mean - margin_of_error
ci_upper = sample_mean + margin_of_error

print(f"Z-score calculated for {confidence_level*100}%: {z_critical:.3f}")
print(f"Mean: {sample_mean:.2f}, CI: [{ci_lower:.2f}, {ci_upper:.2f}]")

Z-score calculated for 95.0%: 1.960
Mean: 346.86, CI: [88.31, 605.40]


<!--- END SOLUTION --->

3. Now repeat the procedure 10000 times (use a for loop ;-) ) and store the values
of the sample mean and the confidence intervals in 2 vectors

<!--- BEGIN SOLUTION --->

In [12]:
mean = []
cis = []  

for i in range(1000):
    sample = marketing_data['history'].sample(20)
    sample_mean = sample.mean()
    sample_std = sample.std()
    
    standard_error = sample_std / np.sqrt(20)
    margin_of_error = z_critical * standard_error
    
    ci_lower = sample_mean - margin_of_error
    ci_upper = sample_mean + margin_of_error
    
    mean.append(sample_mean)
    cis.append((ci_lower, ci_upper))

<!--- END SOLUTION --->

4. Compute the mean of the full history column

<!--- BEGIN SOLUTION --->

In [14]:
# your code here

true_mean = marketing_data['history'].mean()
print(f"True mean: {true_mean:.4f}")

True mean: 242.0857


<!--- END SOLUTION --->


5. How many of the sample means are closer to the actual mean than their respective
confidence interval? Is it what you expected? Explain the result.

Different wording of what should be checked:

If working with margin of error: how often is the distance from sample mean to true mean ≤ margin of error

If working with CIs: How often does the true mean lie within calculated CI?

<!--- BEGIN SOLUTION --->

In [16]:
# your code here


true_mean = marketing_data['history'].mean()

hits = 0
for lower, upper in cis:
    if lower <= true_mean <= upper:
        hits += 1

success_rate = (hits / len(cis)) * 100
print(f"The true mean was captured {success_rate}% of the time.")

The true mean was captured 90.3% of the time.


# Part 2: Bootstrapping

Repeat the exercise done in part 1, but instead of applying formulas, implement
the bootstrap method.

1. Sample 20 values from the history column

In [22]:
initial_sample = marketing_data['history'].sample(20)
print(f"Initial sample mean: {initial_sample.mean():.2f}")

Initial sample mean: 334.20


<!--- END SOLUTION --->

2. Use this sample to compute the sample mean and 95 confidence interval with
   the bootstrap method
   
   **Hints:**
   - The function `df['column_name'].sample(x,replace=T)` returns a series that is the
   same size as x, and that contains sample of x (with replacement).
   - You can use a for loop or list comprehension to sample the data multiple times and store your results in a list.
   - `numpy.percentile(means, 95, interpolation = 'midpoint')` will give you the 95% percentile of the array `means`.
   - The more runs you do, the more computationally expensive: when resampling many times, use vectorization (with `numpy` arrays).
   - numpy is fast when working on an entire array at once, but slow when accessing one element at a time (e.g. in a loop). As such, you might want to create the entire array first, and then calculate the mean of e.g. each row.
   - `np.random.choice()` has a `size` argument that you can pass a tuple to with `(sample_size * number_of_iterations)` .

<!--- BEGIN SOLUTION --->

In [30]:
bootstrap_means = []

for i in range(1000):
    resample = initial_sample.sample(20, replace=True)
    bootstrap_means.append(resample.mean())

ci_lower = np.percentile(bootstrap_means, 2.5)
ci_upper = np.percentile(bootstrap_means, 97.5)

print(f"95% Bootstrap CI: [{ci_lower:.2f}, {ci_upper:.2f}]")

95% Bootstrap CI: [201.00, 476.42]


<!--- END SOLUTION --->

3. Now repeat the procedure 10000 times (hint: use numpy's vector operations, not a for loop) and store the values
of the sample mean and the confidence intervals in 3 numpy arrays or lists

<!--- BEGIN SOLUTION --->

In [31]:
# your code here

all_bootstrap_means = []
all_bootstrap_cis = []

for i in range(10000):
    s = marketing_data['history'].sample(20).values
    
    resamples = np.random.choice(s, size=(1000, 20), replace=True)
    resample_means = resamples.mean(axis=1)
    
    all_bootstrap_means.append(s.mean())
    all_bootstrap_cis.append((np.percentile(resample_means, 2.5), 
                              np.percentile(resample_means, 97.5)))

4. How many of the sample means are closer to the actual mean than their respective confidence interval? Is it what you expected? Explain the result.

In [32]:
# your code here

true_mean = marketing_data['history'].mean()

hits = 0
for lower, upper in all_bootstrap_cis:
    if lower <= true_mean <= upper:
        hits += 1   

success_rate = (hits / len(all_bootstrap_cis)) * 100
print(f"The true mean captured {success_rate}%")


The true mean captured 89.02%
